# 05 — Replicating Fama-French SMB and HML from WRDS/JKP data

This notebook is written as a public, audit-friendly replication of the classic Fama-French 2x3 portfolio sorts. The goal is not just to compute `SMB` and `HML`, but to make every design choice explicit enough that another researcher, recruiter, or reviewer can follow the logic line by line.

## Research question

Can we recover the economic behavior of the official Fama-French size and value factors using a modern stock-level WRDS dataset instead of hand-building the original CRSP/Compustat pipeline?

## Dataset used here

We use the **Jensen, Kelly, and Pedersen (2023) global factor dataset** from WRDS (`contrib.global_factor`), filtered to US equities. This dataset is a good choice for a public replication notebook because it already contains the core inputs we need:

- `market_equity`: firm size, used both for the size sort and for value weights,
- `be_me`: book-to-market, used for the value sort,
- `ret_exc_lead1m`: the stock's excess return in the next month,
- `primary_sec`, `exch_main`, and `source_crsp`: practical filters that let us approximate the original Fama-French universe.

That choice is deliberate. A public repo should minimize hidden data engineering. JKP lets us explain the portfolio logic clearly, while still being honest that it is an approximation to the original CRSP/Compustat construction.

## What Fama and French do

Following Fama and French (1993), we will:

1. build an eligible US common-stock universe,
2. compute **NYSE-only** June breakpoints,
3. sort stocks into six portfolios: `SL`, `SM`, `SH`, `BL`, `BM`, `BH`,
4. value-weight the portfolio returns,
5. aggregate those six returns into `SMB` and `HML`,
6. compare our series with the official Ken French data library.

## Important timing note

The most important implementation detail in this notebook is the return timing. In this repo, `ret_exc_lead1m` is a **lead return**: the row dated June contains the return realized from June to July. That means June rows must map to the **current** June sort year, not the previous one. This notebook makes that convention explicit so the July-to-June holding period is implemented correctly.

## References

- Fama, Eugene F., and Kenneth R. French (1992), *The Cross-Section of Expected Stock Returns*.
- Fama, Eugene F., and Kenneth R. French (1993), *Common Risk Factors in the Returns on Stocks and Bonds*.
- Jensen, Theis I., Bryan T. Kelly, and Lasse H. Pedersen (2023), *Is There a Replication Crisis in Finance?*.

In [ ]:
from pathlib import Path
import os
import sys

# Keep the notebook runnable from either the repo root or the notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Allow an environment-variable override, but default to the public repo
# convention documented in README.md and data/README.md.
default_csv = PROJECT_ROOT / "data" / "jkp_data.csv"
JKP_CSV_PATH = Path(os.environ.get("JKP_CSV_PATH", str(default_csv)))

print(f"Project root: {PROJECT_ROOT}")
print(f"JKP CSV path: {JKP_CSV_PATH}")

if not JKP_CSV_PATH.exists():
    raise FileNotFoundError(
        "Expected a WRDS/JKP extract at "
        f"{JKP_CSV_PATH}. Put your file at data/jkp_data.csv or set JKP_CSV_PATH."
    )

print("Data file found. Ready to load the stock-level panel.")

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from scipy import stats
from statsmodels.tsa.stattools import coint

from quant_trading.data import load_jkp_csv
from quant_trading.evaluation import perf_stats_annualized
from quant_trading.factors import (
    assign_ff_portfolios,
    attach_monthly_portfolios,
    compute_nyse_breakpoints,
    compute_value_weighted_portfolio_returns,
    construct_smb_hml,
    fetch_ff5_factors,
    prepare_ff_universe,
)

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:,.4f}".format

## 1. Load the stock-level panel and define the replication universe

The original Fama-French papers work with common stocks and then apply additional data-availability rules before sorting. We follow the same logic as closely as the public JKP data allows.

### Why start with the universe?

Because the factor returns are a consequence of the *sorting universe*. If the universe changes, the breakpoints change; if the breakpoints change, the factor returns change. That is why this step comes before any portfolio math.

### Filters implemented here

- `primary_sec == 1`: keep the primary security for each firm,
- `exch_main in {1, 2, 3}`: keep NYSE, AMEX, and NASDAQ,
- `source_crsp == 1`: stay close to the CRSP-backed sample,
- `market_equity > 0` and `be_me > 0`: avoid meaningless sorts,
- at least 24 prior monthly observations: transparent proxy for the original "enough accounting history" requirement.

The 24-month history rule is not a claim that Fama and French literally count months this way. It is an explicit approximation that keeps the notebook honest about what is exact and what is adapted to the available data.

In [ ]:
# Load the raw stock-month panel exactly once so every later step starts from
# the same source data.
df_raw = load_jkp_csv(str(JKP_CSV_PATH))

print(f"Raw rows: {len(df_raw):,}")
print(
    f"Raw date range: {df_raw['month_date'].min().date()} "
    f"to {df_raw['month_date'].max().date()}"
)

columns_we_use = [
    "id",
    "month_date",
    "primary_sec",
    "exch_main",
    "source_crsp",
    "market_equity",
    "be_me",
    "ret_exc_lead1m",
]
print("\nColumns used in this notebook:")
print(columns_we_use)

display(df_raw[columns_we_use].head())

In [ ]:
# `prepare_ff_universe` centralizes the rules so the notebook reads like the
# paper: define the eligible universe first, then sort it.
df_ff = prepare_ff_universe(df_raw, min_history_months=24)

print(f"Eligible stock-months: {len(df_ff):,}")
print(f"Unique stocks: {df_ff['id'].nunique():,}")
print(
    f"Sample window: {df_ff['month_date'].min().date()} "
    f"to {df_ff['month_date'].max().date()}"
)

print("\nExchange mix after filtering:")
display(
    df_ff["exch_main"]
    .value_counts()
    .sort_index()
    .rename_axis("exch_main")
    .to_frame("count")
)

display(
    df_ff[
        [
            "id",
            "month_date",
            "market_equity",
            "be_me",
            "obs_count",
            "year",
            "month",
        ]
    ].head()
)

## 2. Recreate the annual June sort

This is the heart of the Fama-French design. Each June, stocks are sorted on size and book-to-market, and that single June classification determines the portfolio membership for the next twelve monthly returns.

### Why NYSE-only breakpoints?

Fama and French use NYSE firms to define the cutoffs because the full cross-section includes many very small firms, especially on NASDAQ. If we let the smallest firms determine the breakpoints, the size and value partitions drift away from the benchmark used in the original papers.

### Why a once-per-year rebalance?

Book equity is an accounting variable, not a daily market price. Annual June rebalancing reduces look-ahead risk and keeps the implementation aligned with the slow-moving accounting information used in the original factor construction.

In [ ]:
# Compute the annual June breakpoints from NYSE stocks only.
breaks = compute_nyse_breakpoints(df_ff)

print(f"Breakpoint years available: {len(breaks)}")
display(breaks.head())

# These are the cutoffs that every exchange will use for that June sort.
print("\nMost recent breakpoint year:")
display(breaks.tail(1))

In [ ]:
# Apply the NYSE breakpoints to all eligible stocks in each June cross-section.
june_assignments = assign_ff_portfolios(df_ff, breaks)

print(f"June stock-year assignments: {len(june_assignments):,}")
print("\nPortfolio counts pooled across all June sorts:")
display(june_assignments["portfolio"].value_counts().sort_index().to_frame("count"))

display(june_assignments.head())

In [ ]:
# Carry each June portfolio label into the monthly holding period.
# Because `ret_exc_lead1m` is a lead return, June month-dated rows already
# represent July realized returns and therefore belong to the current sort year.
df_monthly = attach_monthly_portfolios(df_ff, june_assignments)

print(f"Monthly rows with portfolio labels: {len(df_monthly):,}")
print(f"Stocks represented: {df_monthly['id'].nunique():,}")
print(f"Portfolios present: {sorted(df_monthly['portfolio'].unique())}")

display(
    df_monthly[
        [
            "id",
            "month_date",
            "ffyear",
            "sort_year",
            "portfolio",
            "market_equity",
            "ret_exc_lead1m",
        ]
    ].head()
)

## 3. Convert the six portfolios into monthly returns and factors

At this point every stock-month has a portfolio label. The next step is to aggregate stock returns into portfolio returns.

### Why value weight?

The original Fama-French factors are value-weighted. That choice matters because equal weighting would mechanically lean harder into microcaps and produce a different factor.

### Why shift the dates forward by one month?

Our return field is `ret_exc_lead1m`, so the return stored on a June row is the July return. After we compute the weighted average at June month-date, we shift the index to July so the time stamp matches when the return was actually realized.

In [ ]:
# Aggregate stock returns into the six Fama-French 2x3 portfolios.
port_rets = compute_value_weighted_portfolio_returns(df_monthly)

print(f"Portfolio return months: {len(port_rets)}")
print(
    f"Realized return window: {port_rets.index.min().date()} "
    f"to {port_rets.index.max().date()}"
)

display(port_rets.head())

In [ ]:
# Translate the six base portfolios into the factor returns defined by
# Fama and French (1993).
factors = construct_smb_hml(port_rets)

print(f"Constructed factor months: {len(factors)}")
print("\nMonthly summary statistics:")
display(factors.describe().round(4))

## 4. Compare the reconstructed factors with the official library

A replication is only useful if we can benchmark it against the official Fama-French series. The right question is **not** "did we match every decimal place?" The right question is whether the reconstructed factors capture the same economic variation as the benchmark series.

That is why we will look at several diagnostics rather than only one correlation number:

- direct time-series overlap,
- regression beta and `R^2`,
- rolling correlation,
- annualized tracking error.

If the replication is credible, SMB should usually match more closely than HML because value is especially sensitive to accounting definitions and timing conventions.

In [ ]:
# Pull the official monthly 5-factor file and keep the two factors we need.
# Using the official library as the benchmark makes the comparison transparent.
ff_official = fetch_ff5_factors(
    start_date=str(factors.index.min().date()),
    end_date=str(factors.index.max().date()),
)

print(f"Official FF rows downloaded: {len(ff_official)}")
display(ff_official[["SMB", "HML"]].head())

In [ ]:
# Align both datasets to the same monthly timestamp convention before merging.
our = factors.copy()
our.index = pd.to_datetime(our.index).to_period("M").to_timestamp()

ff_off = ff_official[["SMB", "HML"]].copy()
ff_off.index = pd.to_datetime(ff_off.index).to_period("M").to_timestamp()

comparison = (
    our.rename(columns={"SMB": "Our SMB", "HML": "Our HML"})
    .join(ff_off.rename(columns={"SMB": "FF SMB", "HML": "FF HML"}), how="inner")
    .dropna()
)

print(f"Overlapping months: {len(comparison)}")
print(f"Comparison window: {comparison.index.min().date()} to {comparison.index.max().date()}")
display(comparison.head())

## 5. Read the replication through simple diagnostics

Each diagnostic answers a different question:

- **Correlation**: do the two series move together month by month?
- **Beta from `Our = alpha + beta * Official`**: is our factor scaled like the benchmark?
- **`R^2`**: how much of our factor variation is explained by the official one?
- **Cointegration**: do the two series share a long-run relationship rather than drifting apart permanently?

No single metric is sufficient by itself. For example, a high correlation with a beta far from 1.0 would tell us that we got the direction right but not the scale.

In [ ]:
def compare_factor(our_col, ff_col):
    """Return a compact diagnostic table for one replicated factor."""
    y = comparison[our_col].astype(float)
    x = comparison[ff_col].astype(float)

    corr, corr_p = stats.pearsonr(y, x)
    model = sm.OLS(y, sm.add_constant(x)).fit()
    coint_stat, coint_p, _ = coint(y, x)

    return {
        "correlation": corr,
        "correlation_pvalue": corr_p,
        "alpha_monthly": model.params.iloc[0],
        "beta": model.params.iloc[1],
        "r_squared": model.rsquared,
        "cointegration_stat": coint_stat,
        "cointegration_pvalue": coint_p,
    }


comparison_stats = pd.DataFrame(
    {
        "SMB": compare_factor("Our SMB", "FF SMB"),
        "HML": compare_factor("Our HML", "FF HML"),
    }
).T

display(comparison_stats.round(4))

## 6. Visual diagnostics

Plots are useful because they reveal different failure modes than summary statistics.

- The cumulative-return lines show whether the reconstructed factor drifts away over long horizons.
- The scatter plot shows whether the relation is roughly one-for-one or only directional.
- The rolling correlation shows whether the match is stable or only concentrated in one subperiod.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, (factor, our_col, ff_col) in enumerate(
    [("SMB", "Our SMB", "FF SMB"), ("HML", "Our HML", "FF HML")]
):
    ax = axes[i, 0]
    cum_our = (1 + comparison[our_col]).cumprod()
    cum_ff = (1 + comparison[ff_col]).cumprod()
    ax.plot(comparison.index, cum_our, label=f"Our {factor}", linewidth=2)
    ax.plot(
        comparison.index,
        cum_ff,
        label=f"Official {factor}",
        linestyle="--",
        linewidth=2,
        alpha=0.8,
    )
    ax.set_title(f"Cumulative growth of $1: {factor}")
    ax.set_ylabel("Growth of $1")
    ax.legend()

    ax = axes[i, 1]
    ax.scatter(comparison[ff_col], comparison[our_col], alpha=0.35, s=12)
    lims = [
        min(comparison[ff_col].min(), comparison[our_col].min()),
        max(comparison[ff_col].max(), comparison[our_col].max()),
    ]
    ax.plot(lims, lims, "r--", alpha=0.6, label="45-degree line")
    ax.set_xlabel(f"Official {factor}")
    ax.set_ylabel(f"Our {factor}")
    ax.set_title(f"Scatter comparison: {factor}")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, (factor, our_col, ff_col) in enumerate(
    [("SMB", "Our SMB", "FF SMB"), ("HML", "Our HML", "FF HML")]
):
    rolling_corr = comparison[our_col].rolling(60).corr(comparison[ff_col])
    ax = axes[i]
    ax.plot(comparison.index, rolling_corr, linewidth=1.5)
    ax.axhline(0.9, color="green", linestyle="--", alpha=0.5, label="0.90")
    ax.axhline(0.8, color="orange", linestyle="--", alpha=0.5, label="0.80")
    ax.set_title(f"60-month rolling correlation: {factor}")
    ax.set_ylabel("Correlation")
    ax.set_ylim(0.4, 1.05)
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Performance summary

This table is not meant to "score" the official factors against the reconstructed ones. Instead, it answers a practical question: if someone used the replicated series in downstream research, would the risk/return profile look broadly similar to the benchmark?

In [ ]:
summary = pd.DataFrame(
    {
        col: perf_stats_annualized(comparison[col])
        for col in ["Our SMB", "FF SMB", "Our HML", "FF HML"]
    }
).T

display(
    summary.style.format(
        {
            "Annualized Return": "{:.2%}",
            "Annualized Vol": "{:.2%}",
            "Sharpe": "{:.3f}",
            "Max Drawdown": "{:.2%}",
        }
    )
)

## 8. Tracking error analysis

Tracking error translates the replication gap into a portfolio language. If the annualized tracking error is modest and the mean difference is small, then the reconstructed factor is usually close enough for exploratory research, teaching, and portfolio experiments.

If the tracking error is large, the diagnostics from the previous sections help explain whether the issue is scale, timing, universe definition, or accounting inputs.

In [ ]:
tracking_error = []
for factor in ["SMB", "HML"]:
    diff = comparison[f"Our {factor}"] - comparison[f"FF {factor}"]
    tracking_error.append(
        {
            "factor": factor,
            "annualized_tracking_error": diff.std(ddof=1) * np.sqrt(12),
            "annualized_mean_difference": diff.mean() * 12,
        }
    )

display(pd.DataFrame(tracking_error).set_index("factor").round(4))

## Notes on expected differences and honest interpretation

A strong replication should be **economically close**, not numerically identical. That distinction matters in a public repo because readers should know exactly where approximation enters the construction.

### Why our series should differ from the official library

1. **Book equity definition**: the Ken French library follows a specific accounting hierarchy; `be_me` in JKP is standardized but not literally identical.
2. **Market equity aggregation**: the original library uses CRSP conventions for share classes and firm aggregation; JKP uses its own preprocessed market equity field.
3. **Delisting treatment**: the official series is tied to the Fama-French CRSP workflow, while JKP provides a ready-to-use next-month excess return variable.
4. **Weight evolution within the holding year**: this notebook uses beginning-of-period market equity from the same monthly row; the original library has its own precise lag structure.
5. **Data timing**: Fama-French are careful about when accounting information becomes investable. JKP is designed to be clean and standardized, but its production timing is still a separate implementation.

### What counts as success here?

For a resume-quality public project, success means:

- the code follows the textbook methodology transparently,
- the timing conventions are explicit and tested,
- the differences from the official library are explained rather than hidden,
- the empirical overlap with the official factors is good enough to show that the reconstruction captures the same economic idea.

That is the standard this repo is aiming for.